# EP03. Graph RAG

## 학습 목표

이 노트북을 완료하면 다음을 할 수 있습니다:

1. LangChain `LLMGraphTransformer`로 텍스트에서 **엔티티-관계 자동 추출**
2. `NetworkX`로 **지식 그래프(Knowledge Graph)** 구축 및 시각화
3. **BFS 기반 N-hop 그래프 순회** 함수 구현
4. **그래프 순회 + 벡터 검색** 결합 (`GraphRAGRetriever` 클래스)
5. 일반 RAG vs Graph RAG **다단계 관계 질문 비교 실험**
6. Langfuse로 Graph RAG 파이프라인 추적

---

## AI 가이드 안내

> **이 노트북에서 AI를 활용하는 법**
>
> - 그래프 구조가 처음이라면 섹션 5의 시각화 결과를 Claude에게 보여주며 "이 그래프를 보고 어떤 관계가 있는지 설명해줘"라고 물어보세요.
> - `LLMGraphTransformer` 출력이 예상과 다를 때는 `allowed_nodes`, `allowed_relationships` 조합을 Claude와 함께 튜닝하세요.
> - Exercise는 힌트를 먼저 읽고 스스로 시도한 뒤 AI의 리뷰를 받는 방식을 권장합니다.

**예상 소요 시간:** 약 90~120분  
**필요 API 키:** `ANTHROPIC_API_KEY` (필수), `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` (선택)

---

## 섹션 1. 환경 설정

필요한 패키지를 설치합니다.

In [ ]:
import subprocess, sys

packages = [
    "langchain",
    "langchain-anthropic",
    "langchain-community",
    "langchain-experimental",
    "langchain-chroma",
    "langfuse",
    "networkx",
    "chromadb",
    "sentence-transformers",
    "python-dotenv",
    "matplotlib",
    "pandas",
]

try:
    subprocess.run(["uv", "--version"], check=True, capture_output=True)
    cmd = ["uv", "pip", "install"] + packages
    print("uv를 사용하여 패키지를 설치합니다...")
except (subprocess.CalledProcessError, FileNotFoundError):
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + packages
    print("pip을 사용하여 패키지를 설치합니다...")

result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode == 0:
    print("설치 완료!")
else:
    print("설치 오류:\n", result.stderr[:500])

---

## 섹션 2. 라이브러리 로드 + API 키 설정

In [ ]:
import os
import time
import warnings
warnings.filterwarnings("ignore")

from collections import deque
from typing import List, Dict, Set, Tuple, Optional

import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

from dotenv import load_dotenv

from langchain.schema import Document
from langchain_anthropic import ChatAnthropic
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

load_dotenv()

keys = {
    "ANTHROPIC_API_KEY": os.getenv("ANTHROPIC_API_KEY"),
    "LANGFUSE_PUBLIC_KEY": os.getenv("LANGFUSE_PUBLIC_KEY"),
    "LANGFUSE_SECRET_KEY": os.getenv("LANGFUSE_SECRET_KEY"),
}

for k, v in keys.items():
    status = "설정됨" if v else "미설정"
    print(f"{k}: {status}")

USE_LLM = bool(os.getenv("ANTHROPIC_API_KEY"))
USE_LANGFUSE = bool(os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY"))

print(f"\nLLM (Claude) 사용: {USE_LLM}")
print(f"Langfuse 추적 사용: {USE_LANGFUSE}")

if not USE_LLM:
    print("\n[주의] ANTHROPIC_API_KEY가 없습니다.")
    print("LLMGraphTransformer 섹션은 건너뛰고 사전 정의된 그래프를 사용합니다.")

---

## 섹션 3. 샘플 텍스트 준비 (한국 IT 회사 조직도 + 프로젝트 관계)

In [ ]:
# 한국 IT 회사 조직도 및 프로젝트 관계 텍스트
ORG_TEXTS = [
    """TechCorp의 CTO 이부장은 개발본부 전체를 총괄한다.
개발1팀 팀장 박과장은 이부장에게 직접 보고한다.
개발2팀 팀장 최과장도 이부장에게 직접 보고한다.
인프라팀 팀장 강과장은 이부장 산하에 있다.""",

    """박과장이 이끄는 개발1팀에는 김대리, 이사원, 정사원이 속해 있다.
김대리는 개발1팀 내에서 백엔드 파트 리드를 담당한다.
이사원은 프론트엔드 개발을 담당한다.
정사원은 최근 개발1팀에 합류한 신규 백엔드 개발자다.""",

    """최과장이 이끄는 개발2팀에는 한대리, 오사원이 속해 있다.
한대리는 개발2팀의 시니어 엔지니어로 아키텍처 설계를 담당한다.
오사원은 데이터 엔지니어링을 담당한다.""",

    """강과장이 이끄는 인프라팀에는 윤대리, 신사원이 있다.
윤대리는 Kubernetes 클러스터 운영을 담당한다.
신사원은 CI/CD 파이프라인 관리를 담당한다.""",

    """Alpha 프로젝트는 TechCorp의 핵심 서비스 플랫폼 재구축 프로젝트다.
Alpha 프로젝트는 개발1팀이 주도하며 박과장이 프로젝트 리드를 맡고 있다.
Alpha 프로젝트에는 김대리, 이사원이 풀타임으로 참여한다.
Alpha 프로젝트는 Kubernetes, FastAPI, PostgreSQL, Redis를 기술 스택으로 사용한다.""",

    """Beta 프로젝트는 데이터 분석 플랫폼 구축 프로젝트다.
Beta 프로젝트는 개발2팀이 담당하며 최과장이 책임자다.
한대리와 오사원이 Beta 프로젝트에 참여한다.
Beta 프로젝트는 Apache Kafka, Spark, Airflow, React를 사용한다.""",

    """Gamma 프로젝트는 인프라 현대화 프로젝트로 인프라팀이 주도한다.
강과장이 Gamma 프로젝트 책임자이며 윤대리, 신사원이 참여한다.
Gamma 프로젝트는 Terraform, Ansible, Prometheus, Grafana를 사용한다.
Alpha 프로젝트와 Beta 프로젝트 모두 Gamma 프로젝트의 인프라에 의존한다.""",

    """Alpha 프로젝트의 백엔드 서비스는 PostgreSQL 데이터베이스를 주 저장소로 사용한다.
Redis는 Alpha 프로젝트에서 세션 캐시와 메시지 큐 용도로 활용된다.
Alpha 프로젝트의 API 서버는 FastAPI 기반으로 구축되었다.
Kubernetes는 Alpha 프로젝트와 Gamma 프로젝트 양쪽에서 사용된다.""",

    """정사원은 Alpha 프로젝트에 신규 합류하여 PostgreSQL 마이그레이션 태스크를 담당한다.
김대리는 Alpha 프로젝트의 FastAPI 서비스 설계를 리드한다.
이사원이 개발한 React 기반 대시보드는 Alpha 프로젝트의 관리자 페이지다.""",
]

# LangChain Document 객체로 변환
org_documents = [
    Document(page_content=text, metadata={"source": f"org_doc_{i+1}"})
    for i, text in enumerate(ORG_TEXTS)
]

print(f"준비된 문서 수: {len(org_documents)}")
print("\n[문서 샘플 - 첫 번째 문서]")
print(org_documents[0].page_content)

# 다단계 관계 테스트 쿼리
GRAPH_TEST_QUERIES = [
    {
        "query": "이부장이 관리하는 모든 팀원의 이름은 무엇인가요?",
        "type": "2-hop",
        "expected_entities": ["박과장", "최과장", "강과장", "김대리", "이사원", "정사원", "한대리", "오사원", "윤대리", "신사원"]
    },
    {
        "query": "Alpha 프로젝트에 참여하는 팀원들이 사용하는 기술 스택은 무엇인가요?",
        "type": "2-hop",
        "expected_entities": ["Kubernetes", "FastAPI", "PostgreSQL", "Redis"]
    },
    {
        "query": "박과장 팀에서 데이터베이스 관련 작업을 하는 팀원은 누구인가요?",
        "type": "3-hop",
        "expected_entities": ["정사원", "김대리"]
    },
]

print(f"\n테스트 쿼리 수: {len(GRAPH_TEST_QUERIES)}")

---

## 섹션 4. LLMGraphTransformer로 엔티티-관계 추출

In [ ]:
if USE_LLM:
    from langchain_experimental.graph_transformers import LLMGraphTransformer
    
    llm = ChatAnthropic(
        model="claude-3-5-sonnet-20241022",
        temperature=0
    )
    
    transformer = LLMGraphTransformer(
        llm=llm,
        allowed_nodes=["사람", "팀", "프로젝트", "기술", "회사"],
        allowed_relationships=[
            "소속", "보고", "관리", "담당", "참여", "사용기술", "의존", "리드"
        ],
        strict_mode=False
    )
    
    print("LLMGraphTransformer 초기화 완료")
    print("엔티티-관계 추출 중... (Claude API 호출, 약 30~60초 소요)")
    
    start_time = time.perf_counter()
    graph_documents = transformer.convert_to_graph_documents(org_documents)
    elapsed = time.perf_counter() - start_time
    
    print(f"추출 완료! ({elapsed:.1f}초)")
    print(f"\n처리된 문서 수: {len(graph_documents)}")
    
    total_nodes = sum(len(gd.nodes) for gd in graph_documents)
    total_rels = sum(len(gd.relationships) for gd in graph_documents)
    print(f"총 추출된 노드: {total_nodes}개")
    print(f"총 추출된 관계: {total_rels}개")
    
    print("\n[첫 번째 문서 추출 결과 샘플]")
    gd = graph_documents[0]
    print("노드:")
    for n in gd.nodes:
        print(f"  - {n.id} (타입: {n.type})")
    print("관계:")
    for r in gd.relationships:
        print(f"  - {r.source.id} --[{r.type}]--> {r.target.id}")
else:
    print("[주의] ANTHROPIC_API_KEY가 없어 LLMGraphTransformer를 건너뜁니다.")
    print("섹션 5에서 사전 정의된 그래프를 직접 구축합니다.")
    graph_documents = []

---

## 섹션 5. NetworkX 그래프 구축 및 시각화

**[Graph RAG 아키텍처]**
```mermaid
flowchart TD
    T[/"📄 원문 텍스트"\]:::doc --> E("🤖 엔티티/관계 추출<br/>LLMGraphTransformer"):::extract
    E --> G[("🕸️ 지식 그래프 구축<br/>NetworkX")]:::graph
    G --> GT("🏃 그래프 순회<br/>BFS (1-hop, 2-hop)"):::traverse
    Q((사용자<br/>쿼리)):::query --> VS("🔍 벡터 검색<br/>ChromaDB"):::vector
    Q --> GT
    GT --> M{"🔗 컨텍스트 병합<br/>(그래프 + 벡터문서)"}:::merge
    VS --> M
    M --> LLM([✨ LLM 답변 생성<br/>Claude / GPT]):::llm
    classDef doc fill:#EA4335,stroke:#fff,color:#fff,stroke-width:2px,rx:5px,ry:5px
    classDef extract fill:#4285F4,stroke:#fff,color:#fff,stroke-width:2px,rx:5px,ry:5px
    classDef graph fill:#FBBC05,stroke:#fff,color:#000,stroke-width:2px,rx:5px,ry:5px
    classDef traverse fill:#8e24aa,stroke:#fff,color:#fff,stroke-width:2px,rx:5px,ry:5px
    classDef query fill:#607d8b,stroke:#fff,color:#fff,stroke-width:2px,font-weight:bold
    classDef vector fill:#26a69a,stroke:#fff,color:#fff,stroke-width:2px,rx:5px,ry:5px
    classDef merge fill:#ff7043,stroke:#fff,color:#fff,stroke-width:2px,font-weight:bold
    classDef llm fill:#5c6bc0,stroke:#fff,color:#fff,stroke-width:2px,rx:5px,ry:5px
```


In [ ]:
# NetworkX 방향 그래프 초기화
G = nx.DiGraph()

if graph_documents:
    # LLMGraphTransformer 결과를 NetworkX 그래프로 변환
    for gd in graph_documents:
        for node in gd.nodes:
            G.add_node(node.id, node_type=node.type, **node.properties)
        for rel in gd.relationships:
            G.add_edge(
                rel.source.id,
                rel.target.id,
                relation=rel.type,
                **rel.properties
            )
    print(f"LLMGraphTransformer 결과로 그래프 구축")
else:
    # API 키 없을 때 사용할 사전 정의 그래프
    print("사전 정의된 그래프를 구축합니다...")
    
    # 노드 추가
    people = {
        "이부장": "사람", "박과장": "사람", "최과장": "사람", "강과장": "사람",
        "김대리": "사람", "이사원": "사람", "정사원": "사람",
        "한대리": "사람", "오사원": "사람",
        "윤대리": "사람", "신사원": "사람"
    }
    teams = {"개발1팀": "팀", "개발2팀": "팀", "인프라팀": "팀"}
    projects = {"Alpha프로젝트": "프로젝트", "Beta프로젝트": "프로젝트", "Gamma프로젝트": "프로젝트"}
    techs = {
        "Kubernetes": "기술", "FastAPI": "기술", "PostgreSQL": "기술",
        "Redis": "기술", "Kafka": "기술", "Spark": "기술",
        "React": "기술", "Terraform": "기술", "Prometheus": "기술"
    }
    
    for name, ntype in {**people, **teams, **projects, **techs}.items():
        G.add_node(name, node_type=ntype)
    
    # 관계 추가
    edges = [
        # 보고 관계
        ("박과장", "이부장", "보고"), ("최과장", "이부장", "보고"), ("강과장", "이부장", "보고"),
        # 소속 관계
        ("박과장", "개발1팀", "관리"), ("최과장", "개발2팀", "관리"), ("강과장", "인프라팀", "관리"),
        ("김대리", "개발1팀", "소속"), ("이사원", "개발1팀", "소속"), ("정사원", "개발1팀", "소속"),
        ("한대리", "개발2팀", "소속"), ("오사원", "개발2팀", "소속"),
        ("윤대리", "인프라팀", "소속"), ("신사원", "인프라팀", "소속"),
        # 프로젝트 담당
        ("개발1팀", "Alpha프로젝트", "담당"), ("개발2팀", "Beta프로젝트", "담당"),
        ("인프라팀", "Gamma프로젝트", "담당"),
        ("박과장", "Alpha프로젝트", "리드"), ("최과장", "Beta프로젝트", "리드"),
        ("강과장", "Gamma프로젝트", "리드"),
        ("김대리", "Alpha프로젝트", "참여"), ("이사원", "Alpha프로젝트", "참여"),
        ("정사원", "Alpha프로젝트", "참여"),
        ("한대리", "Beta프로젝트", "참여"), ("오사원", "Beta프로젝트", "참여"),
        # 기술 사용
        ("Alpha프로젝트", "Kubernetes", "사용기술"), ("Alpha프로젝트", "FastAPI", "사용기술"),
        ("Alpha프로젝트", "PostgreSQL", "사용기술"), ("Alpha프로젝트", "Redis", "사용기술"),
        ("Beta프로젝트", "Kafka", "사용기술"), ("Beta프로젝트", "Spark", "사용기술"),
        ("Beta프로젝트", "React", "사용기술"),
        ("Gamma프로젝트", "Terraform", "사용기술"), ("Gamma프로젝트", "Kubernetes", "사용기술"),
        ("Gamma프로젝트", "Prometheus", "사용기술"),
        # 의존 관계
        ("Alpha프로젝트", "Gamma프로젝트", "의존"), ("Beta프로젝트", "Gamma프로젝트", "의존"),
    ]
    
    for src, tgt, rel in edges:
        G.add_edge(src, tgt, relation=rel)

print(f"그래프 구축 완료")
print(f"  노드 수: {G.number_of_nodes()}")
print(f"  엣지 수: {G.number_of_edges()}")

# 노드 타입별 색상 정의 및 시각화
import platform
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Linux':
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

type_colors = {
    "사람": "#93c5fd",
    "팀": "#86efac",
    "프로젝트": "#fde68a",
    "기술": "#f9a8d4",
    "회사": "#c4b5fd"
}

def get_node_color(node):
    ntype = G.nodes[node].get("node_type", "기타")
    return type_colors.get(ntype, "#e5e7eb")

node_colors = [get_node_color(n) for n in G.nodes()]

fig, ax = plt.subplots(1, 1, figsize=(16, 10))
pos = nx.spring_layout(G, k=2.5, seed=42, iterations=50)

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1200, ax=ax, alpha=0.9)
nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold', ax=ax)
nx.draw_networkx_edges(G, pos, edge_color='#6b7280', arrows=True,
                       arrowsize=15, width=1.2, ax=ax, alpha=0.7,
                       connectionstyle='arc3,rad=0.1')
edge_labels = nx.get_edge_attributes(G, 'relation')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels,
                              font_size=6, font_color='#374151', ax=ax)

legend_patches = [mpatches.Patch(color=color, label=ntype)
                  for ntype, color in type_colors.items()]
ax.legend(handles=legend_patches, loc='upper left', fontsize=9)

ax.set_title("TechCorp 조직도 & 프로젝트 지식 그래프", fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('ep03_knowledge_graph.png', dpi=150, bbox_inches='tight')
plt.show()
print("그래프 저장 완료: ep03_knowledge_graph.png")

---

## 섹션 6. 그래프 통계 분석 (노드 수, 엣지 수, 중심성)

In [ ]:
print("=" * 50)
print("그래프 통계 분석")
print("=" * 50)

# 기본 통계
print(f"\n기본 통계:")
print(f"  총 노드 수: {G.number_of_nodes()}")
print(f"  총 엣지 수: {G.number_of_edges()}")
print(f"  평균 차수: {sum(dict(G.degree()).values()) / G.number_of_nodes():.2f}")

# 노드 타입별 분포
type_counts = {}
for node in G.nodes():
    ntype = G.nodes[node].get('node_type', '기타')
    type_counts[ntype] = type_counts.get(ntype, 0) + 1

print(f"\n노드 타입별 분포:")
for ntype, count in sorted(type_counts.items()):
    print(f"  {ntype}: {count}개")

# 관계 타입별 분포
rel_counts = {}
for _, _, data in G.edges(data=True):
    rel = data.get('relation', '기타')
    rel_counts[rel] = rel_counts.get(rel, 0) + 1

print(f"\n관계 타입별 분포:")
for rel, count in sorted(rel_counts.items(), key=lambda x: -x[1]):
    print(f"  {rel}: {count}개")

# 중심성 분석 (In-degree: 많이 참조되는 노드)
in_degree = dict(G.in_degree())
out_degree = dict(G.out_degree())

top_in_degree = sorted(in_degree.items(), key=lambda x: -x[1])[:5]
top_out_degree = sorted(out_degree.items(), key=lambda x: -x[1])[:5]

print(f"\n[In-Degree 상위 5 - 많이 참조되는 노드]:")
for node, deg in top_in_degree:
    ntype = G.nodes[node].get('node_type', '?')
    print(f"  {node} ({ntype}): {deg}")

print(f"\n[Out-Degree 상위 5 - 많이 연결하는 노드]:")
for node, deg in top_out_degree:
    ntype = G.nodes[node].get('node_type', '?')
    print(f"  {node} ({ntype}): {deg}")

# PageRank 중심성
try:
    pagerank = nx.pagerank(G, alpha=0.85)
    top_pr = sorted(pagerank.items(), key=lambda x: -x[1])[:5]
    print(f"\n[PageRank 상위 5 - 전체 그래프에서 중요한 노드]:")
    for node, pr in top_pr:
        ntype = G.nodes[node].get('node_type', '?')
        print(f"  {node} ({ntype}): {pr:.4f}")
except Exception as e:
    print(f"PageRank 계산 실패: {e}")

---

## 섹션 7. BFS 기반 그래프 순회 함수 구현

In [ ]:
def bfs_neighbors(
    graph: nx.DiGraph,
    start_node: str,
    max_hop: int = 2,
    direction: str = "both"  # "out", "in", "both"
) -> Dict[str, dict]:
    """
    BFS 기반 N-hop 이웃 탐색
    
    Returns:
        {node_id: {"hop": int, "path": list, "relations": list}}
    """
    if start_node not in graph:
        return {}
    
    visited = {start_node: {"hop": 0, "path": [start_node], "relations": []}}
    queue = deque([(start_node, 0)])
    
    while queue:
        current, hop = queue.popleft()
        
        if hop >= max_hop:
            continue
        
        # 방향에 따른 이웃 노드 탐색
        neighbors = []
        if direction in ("out", "both"):
            for neighbor in graph.successors(current):
                rel = graph[current][neighbor].get('relation', '')
                neighbors.append((neighbor, rel, "out"))
        if direction in ("in", "both"):
            for neighbor in graph.predecessors(current):
                rel = graph[neighbor][current].get('relation', '')
                neighbors.append((neighbor, f"<-{rel}", "in"))
        
        for neighbor, rel, _ in neighbors:
            if neighbor not in visited:
                parent_info = visited[current]
                visited[neighbor] = {
                    "hop": hop + 1,
                    "path": parent_info["path"] + [neighbor],
                    "relations": parent_info["relations"] + [rel]
                }
                queue.append((neighbor, hop + 1))
    
    # 시작 노드 제외하고 반환
    visited.pop(start_node)
    return visited


def graph_context_to_text(graph: nx.DiGraph, neighbors: Dict) -> str:
    """그래프 순회 결과를 텍스트 컨텍스트로 변환"""
    lines = []
    for node_id, info in sorted(neighbors.items(), key=lambda x: x[1]['hop']):
        hop = info['hop']
        path = " -> ".join(info['path'])
        ntype = graph.nodes[node_id].get('node_type', '?') if node_id in graph else '?'
        lines.append(f"[{hop}-hop] {node_id} ({ntype}): 경로 = {path}")
    return "\n".join(lines)


# 단위 테스트
print("BFS 순회 함수 테스트")
print("=" * 50)

# 이부장 기준 2-hop 순회
result_2hop = bfs_neighbors(G, "이부장", max_hop=2, direction="out")
print(f"\n이부장 기준 2-hop 이웃 (방향: 아웃바운드):")
for node, info in sorted(result_2hop.items(), key=lambda x: x[1]['hop']):
    ntype = G.nodes[node].get('node_type', '?') if node in G else '?'
    print(f"  [{info['hop']}-hop] {node} ({ntype})")

# Alpha프로젝트 기준 1-hop 순회
result_alpha = bfs_neighbors(G, "Alpha프로젝트", max_hop=1, direction="out")
print(f"\nAlpha프로젝트 기준 1-hop 이웃 (기술 스택):")
for node, info in result_alpha.items():
    ntype = G.nodes[node].get('node_type', '?') if node in G else '?'
    print(f"  {node} ({ntype}) - 관계: {info['relations']}")

---

## 섹션 8. 1-hop, 2-hop 이웃 검색 실험

In [ ]:
def answer_graph_query(
    graph: nx.DiGraph,
    entity: str,
    max_hop: int = 2,
    direction: str = "both"
) -> str:
    """그래프 순회 결과로 질문 컨텍스트 생성"""
    neighbors = bfs_neighbors(graph, entity, max_hop=max_hop, direction=direction)
    
    if not neighbors:
        return f"{entity}에 대한 그래프 정보를 찾을 수 없습니다."
    
    # 타입별 그룹화
    type_groups = {}
    for node_id, info in neighbors.items():
        ntype = graph.nodes[node_id].get('node_type', '기타') if node_id in graph else '기타'
        if ntype not in type_groups:
            type_groups[ntype] = []
        type_groups[ntype].append({
            "name": node_id,
            "hop": info['hop'],
            "path": " -> ".join(info['path'])
        })
    
    lines = [f"{entity} 기준 {max_hop}-hop 이내 관련 엔티티:\n"]
    for ntype, nodes in type_groups.items():
        lines.append(f"[{ntype}]")
        for node in sorted(nodes, key=lambda x: x['hop']):
            lines.append(f"  - {node['name']} ({node['hop']}-hop): {node['path']}")
        lines.append("")
    
    return "\n".join(lines)


# 세 가지 핵심 엔티티에 대한 그래프 순회 실험
test_cases = [
    ("이부장", 2, "out"),
    ("Alpha프로젝트", 1, "both"),
    ("박과장", 2, "both"),
]

for entity, hop, direction in test_cases:
    print("\n" + "=" * 55)
    print(f"엔티티: {entity} | hop: {hop} | 방향: {direction}")
    print("=" * 55)
    print(answer_graph_query(G, entity, max_hop=hop, direction=direction))

---

## 섹션 9. 벡터 검색 구현 (ChromaDB)

In [ ]:
print("임베딩 모델 로드 중...")
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("ChromaDB 벡터스토어 구축 중...")
vectorstore = Chroma.from_documents(
    documents=org_documents,
    embedding=embedding_model,
    collection_name="ep03_graph_rag_demo"
)

base_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print(f"벡터스토어 구축 완료: {vectorstore._collection.count()}개 문서")

# 벡터 검색 동작 확인
test_query = GRAPH_TEST_QUERIES[1]["query"]
vector_results = base_retriever.invoke(test_query)

print(f"\n[벡터 검색] 쿼리: '{test_query[:50]}...'")
print(f"검색된 문서 수: {len(vector_results)}")
for i, doc in enumerate(vector_results, 1):
    print(f"  {i}. {doc.page_content[:80]}...")

---

## 섹션 10. 그래프 순회 + 벡터 검색 결합 (GraphRAGRetriever)

**[N-hop 그래프 순회 전략]**
```mermaid
flowchart LR
    Q((쿼리: "박과장")):::start
    Q --> H1("1-hop 이웃<br/>(직접 연결)"):::hop1
    H1 -.-> N1[/"Alpha 프로젝트<br/>박과장 팀"\]:::node1
    H1 --> H2("2-hop 이웃<br/>(간접 연결)"):::hop2
    H2 -.-> N2[/"Kubernetes<br/>김대리<br/>이사원"\]:::node2
    H2 --> H3("3-hop... <br/>(노이즈 증가)"):::hop3
    classDef start fill:#e53935,stroke:#fff,color:#fff,stroke-width:2px,font-weight:bold
    classDef hop1 fill:#1e88e5,stroke:#fff,color:#fff,stroke-width:2px,rx:10px,ry:10px
    classDef hop2 fill:#43a047,stroke:#fff,color:#fff,stroke-width:2px,rx:10px,ry:10px
    classDef hop3 fill:#fb8c00,stroke:#fff,color:#fff,stroke-width:2px,rx:10px,ry:10px
    classDef node1 fill:#bbdefb,stroke:#1e88e5,color:#000
    classDef node2 fill:#c8e6c9,stroke:#43a047,color:#000
```


In [ ]:
import re

class GraphRAGRetriever:
    """
    그래프 순회 + 벡터 검색을 결합한 하이브리드 Retriever
    """
    
    def __init__(
        self,
        graph: nx.DiGraph,
        vectorstore,
        max_hop: int = 2,
        vector_k: int = 3,
    ):
        self.graph = graph
        self.vectorstore = vectorstore
        self.max_hop = max_hop
        self.vector_k = vector_k
        self.entity_list = list(graph.nodes())
    
    def _extract_entities_from_query(self, query: str) -> List[str]:
        """쿼리에서 그래프 노드명과 일치하는 엔티티 추출"""
        found = []
        for entity in self.entity_list:
            if entity in query:
                found.append(entity)
        return found
    
    def _graph_context(self, entities: List[str]) -> str:
        """엔티티 목록에 대한 그래프 순회 컨텍스트 생성"""
        if not entities:
            return ""
        
        all_context = []
        seen_nodes = set()
        
        for entity in entities:
            neighbors = bfs_neighbors(self.graph, entity,
                                       max_hop=self.max_hop, direction="both")
            for node_id, info in neighbors.items():
                if node_id not in seen_nodes:
                    seen_nodes.add(node_id)
                    ntype = self.graph.nodes[node_id].get('node_type', '?') \
                            if node_id in self.graph else '?'
                    relations = " -> ".join(info['path'])
                    all_context.append(
                        f"{node_id}({ntype}): {relations}"
                    )
        
        return "[그래프 관계 정보]\n" + "\n".join(all_context)
    
    def retrieve(self, query: str) -> dict:
        """하이브리드 검색 실행"""
        # 1. 쿼리에서 엔티티 추출
        entities = self._extract_entities_from_query(query)
        
        # 2. 그래프 순회 컨텍스트
        graph_ctx = self._graph_context(entities)
        
        # 3. 벡터 검색
        vector_docs = self.vectorstore.similarity_search(query, k=self.vector_k)
        vector_ctx = "\n\n".join(
            f"[문서 {i+1}]\n{doc.page_content}"
            for i, doc in enumerate(vector_docs)
        )
        
        # 4. 컨텍스트 병합
        combined_context = ""
        if graph_ctx:
            combined_context += graph_ctx + "\n\n"
        combined_context += "[원문 문서]\n" + vector_ctx
        
        return {
            "context": combined_context,
            "entities_found": entities,
            "graph_nodes_retrieved": len(graph_ctx.split('\n')) if graph_ctx else 0,
            "vector_docs_retrieved": len(vector_docs)
        }


# GraphRAGRetriever 초기화
graph_rag = GraphRAGRetriever(
    graph=G,
    vectorstore=vectorstore,
    max_hop=2,
    vector_k=3
)

# 테스트
test_q = GRAPH_TEST_QUERIES[0]["query"]
result = graph_rag.retrieve(test_q)

print(f"쿼리: '{test_q}'")
print(f"추출된 엔티티: {result['entities_found']}")
print(f"그래프 노드: {result['graph_nodes_retrieved']}개")
print(f"벡터 문서: {result['vector_docs_retrieved']}개")
print(f"\n통합 컨텍스트 (처음 500자):")
print(result['context'][:500])

---

## 섹션 11. 일반 RAG vs Graph RAG 비교 실험

In [ ]:
if USE_LLM:
    llm = ChatAnthropic(
        model="claude-3-5-haiku-20241022",
        temperature=0,
        max_tokens=512
    )
    
    rag_prompt = ChatPromptTemplate.from_template(
        """다음 컨텍스트 정보를 활용하여 질문에 답하세요.
답변은 컨텍스트에서 찾은 정보만 사용하세요. 모르는 내용은 "정보 없음"이라고 답하세요.

컨텍스트:
{context}

질문: {question}

답변:"""
    )
    
    def run_standard_rag(query: str) -> dict:
        """일반 RAG 실행"""
        start = time.perf_counter()
        vector_docs = vectorstore.similarity_search(query, k=3)
        context = "\n\n".join(doc.page_content for doc in vector_docs)
        answer = (rag_prompt | llm | StrOutputParser()).invoke(
            {"context": context, "question": query}
        )
        latency = (time.perf_counter() - start) * 1000
        return {"answer": answer, "latency_ms": latency, "context_length": len(context)}
    
    def run_graph_rag(query: str) -> dict:
        """Graph RAG 실행"""
        start = time.perf_counter()
        retrieval = graph_rag.retrieve(query)
        answer = (rag_prompt | llm | StrOutputParser()).invoke(
            {"context": retrieval['context'], "question": query}
        )
        latency = (time.perf_counter() - start) * 1000
        return {
            "answer": answer,
            "latency_ms": latency,
            "context_length": len(retrieval['context']),
            "entities_found": retrieval['entities_found']
        }
    
    print("일반 RAG vs Graph RAG 비교 실험")
    print("=" * 60)
    
    comparison_results = []
    
    for q_data in GRAPH_TEST_QUERIES:
        query = q_data["query"]
        qtype = q_data["type"]
        
        print(f"\n[{qtype} 쿼리] {query}")
        print("-" * 50)
        
        std_result = run_standard_rag(query)
        graph_result = run_graph_rag(query)
        
        print(f"[일반 RAG]")
        print(f"  답변: {std_result['answer'][:200]}")
        print(f"  지연: {std_result['latency_ms']:.0f}ms")
        
        print(f"\n[Graph RAG]")
        print(f"  발견 엔티티: {graph_result['entities_found']}")
        print(f"  답변: {graph_result['answer'][:200]}")
        print(f"  지연: {graph_result['latency_ms']:.0f}ms")
        
        comparison_results.append({
            "쿼리": query[:40] + "...",
            "타입": qtype,
            "일반RAG_지연(ms)": int(std_result['latency_ms']),
            "GraphRAG_지연(ms)": int(graph_result['latency_ms']),
            "컨텍스트_비율": f"{len(graph_result['answer'])}/{len(std_result['answer'])}"
        })
    
    print("\n\n결과 요약:")
    print(pd.DataFrame(comparison_results).to_string(index=False))
else:
    print("[주의] ANTHROPIC_API_KEY가 없어 LLM 비교 실험을 건너뜁니다.")
    print("그래프 순회 컨텍스트 생성 테스트만 진행합니다.")
    
    for q_data in GRAPH_TEST_QUERIES:
        print(f"\n쿼리: {q_data['query']}")
        retrieval = graph_rag.retrieve(q_data['query'])
        print(f"추출된 엔티티: {retrieval['entities_found']}")
        print(f"컨텍스트 일부:\n{retrieval['context'][:300]}...")

---

## 섹션 12. Langfuse 통합

In [ ]:
if USE_LANGFUSE and USE_LLM:
    from langfuse.langchain import CallbackHandler
    from langfuse import Langfuse
    
    langfuse_client = Langfuse(
        public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
        secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    )
    
    langfuse_handler = CallbackHandler(
        public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
        secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    )
    
    print("Langfuse 통합 초기화 완료")
    
    # Graph RAG 파이프라인 추적 실행
    trace = langfuse_client.trace(
        name="graph-rag-pipeline",
        metadata={
            "graph_nodes": G.number_of_nodes(),
            "graph_edges": G.number_of_edges(),
            "max_hop": graph_rag.max_hop,
        }
    )
    
    test_query = GRAPH_TEST_QUERIES[0]["query"]
    
    # Graph RAG 검색 span
    retrieval_span = trace.span(name="graph-rag-retrieval")
    retrieval = graph_rag.retrieve(test_query)
    retrieval_span.end(
        output={
            "entities_found": retrieval["entities_found"],
            "graph_nodes_retrieved": retrieval["graph_nodes_retrieved"],
            "context_length": len(retrieval["context"])
        }
    )
    
    # LLM 답변 생성 (Langfuse 콜백 포함)
    llm_chain = rag_prompt | llm | StrOutputParser()
    answer = llm_chain.invoke(
        {"context": retrieval["context"], "question": test_query},
        config={"callbacks": [langfuse_handler]}
    )
    
    trace.update(output={"answer": answer})
    
    print(f"\n추적 완료!")
    print(f"쿼리: {test_query}")
    print(f"답변: {answer[:300]}")
    print(f"\nLangfuse 대시보드에서 확인: https://cloud.langfuse.com")
else:
    msg = []
    if not USE_LANGFUSE:
        msg.append("LANGFUSE 키 미설정")
    if not USE_LLM:
        msg.append("ANTHROPIC_API_KEY 미설정")
    print(f"[주의] Langfuse 통합 건너뜀: {', '.join(msg)}")

---

## Exercise

### Exercise 1: 조직도 텍스트에서 그래프 구축

**목표:** 새로운 조직도 텍스트를 입력으로 지식 그래프를 구축하고 시각화하세요.

**힌트:**
- 기존 `ORG_TEXTS`와 다른 회사/조직 구조를 사용하세요.
- `LLMGraphTransformer` 또는 수동으로 그래프를 구축할 수 있습니다.
- 시각화 시 노드 타입별 색상을 다르게 지정하세요.

In [ ]:
# Exercise 1: 나만의 조직도 텍스트로 그래프 구축

MY_ORG_TEXTS = [
    # TODO: 새로운 조직도 텍스트를 여기에 작성하세요
    """
    여기에 조직도 텍스트를 작성하세요.
    예: FinTech 스타트업, 병원 조직도, 대학 연구실 구조 등
    """
]

# TODO: 그래프 구축 및 시각화
# G_my = nx.DiGraph()
# ...

# 완성 후 체크리스트:
# [ ] 노드 수, 엣지 수 출력
# [ ] 노드 타입별 색상 시각화
# [ ] 중심성 높은 노드 3개 출력
# [ ] 그래프 이미지 저장

### Exercise 2: 그래프 순회로 다단계 관계 질의

**목표:** 구축한 그래프에서 BFS 순회를 이용해 다단계 관계 질문에 답하는 파이프라인을 완성하세요.

**힌트:**
- `bfs_neighbors()` 함수를 활용하세요.
- 일반 벡터 검색과 Graph RAG 답변을 나란히 비교하세요.
- 답변 품질을 주관적으로 평가하고 그 이유를 주석으로 작성하세요.
- 보너스: Langfuse에 각 질의 결과를 로깅하세요.

In [ ]:
# Exercise 2: 다단계 관계 질의 파이프라인

MY_MULTI_HOP_QUERIES = [
    # TODO: 여러분이 만든 그래프에 맞는 다단계 질문 3개 정의
    {
        "query": "2-hop 관계 질문 예시",
        "hop_type": "2-hop",
        "key_entities": ["엔티티명"],
    },
    # ...
]

# TODO: 각 질문에 대해 일반 RAG vs Graph RAG 답변 비교
# for q in MY_MULTI_HOP_QUERIES:
#     print(f"질문: {q['query']}")
#     # 1. 벡터 검색만 사용한 답변
#     # 2. 그래프 순회 + 벡터 검색 답변
#     # 3. 두 답변 비교 및 평가

# 보너스: 결과를 DataFrame으로 정리하고 시각화
print("Exercise 2를 완성하세요!")